# 01 May 05 Default Best Fixed Baseline

Re-runs the best May 04 default DRLB params as the May 05 control point.

In [ ]:
import sys
import json
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess


In [ ]:
RUN_NAME = 'may06_default_best_fixed_baseline'
DRLB_PROFILE = 'may06_default_best_fixed'
VERBOSE = False


In [ ]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
config = replace(config, n_trials=1, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'base_drlb_params': base_drlb_params,
    'reference_model_params': reference_model_params,
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}


[I 2026-05-06 01:12:00,227] A new study created in memory with name: no-name-9e95c012-bd87-4c30-af5a-7387c1f081a4
[I 2026-05-06 01:15:10,316] Trial 0 finished with value: 2321.708352250234 and parameters: {'dqn_lr': 0.0003, 'reward_net_lr': 0.01, 'bid_lower_clip': 3, 'bid_upper_clip': 8}. Best is trial 0 with value: 2321.708352250234.


{'run_name': 'may06_default_best_fixed_baseline',
 'profile': 'may05_default_best_fixed',
 'base_drlb_params': {'max_bid': 100.0,
  'T': 72,
  'lambda_min': -inf,
  'lambda_max': inf,
  'bids_per_timestep': 1,
  'dqn_soft_update_tau': 0.01,
  'dqn_loss_type': 'smooth_l1',
  'dqn_grad_clip_norm': 5.0,
  'dqn_reward_clip_value': 10.0,
  'init_lambda': 0.0028423174374845716,
  'init_lambda_mode': 'constant',
  'bid_lower_clip': 3,
  'bid_upper_clip': 8,
  'traffic_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/traffic_share.csv'},
 'reference_model_params': {'dqn_gamma': 1.0,
  'dqn_lr': 0.0003,
  'dqn_target_update_interval': 100,
  'reward_net_lr': 0.01,
  'dqn_epsilon_start': 0.95,
  'dqn_epsilon_end': 0.05,
  'dqn_epsilon_anneal': 2e-05},
 'best_val_metrics': {'cpc_relative': 507.20785184250104,
  'rmse': 1.4605519264033147,
  'clicks_sum': 2321.708352250234,
  'quickspend': 0.042801556420233464,
  'skipped_campaigns': 0,
  'time_inference_sec': 17.538306236267

In [ ]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])


,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
